Brian Saville
August 3, 2026
An attempt to download iNaturalist data to gather a "Bee Dex" csv for Manhattan and the Bronx.

First, import the necessary functions

In [2]:
import requests
from collections import Counter
import pandas as pd
import time

Now, query the iNat database as needed

In [3]:
url = "https://api.inaturalist.org/v2/observations"

bee_id = 630955
bronx_id = 1189
manhattan_id = 1264
num_obs = 13651

params = {
    "taxon_id": bee_id,
    "quality_grade": "research",
    "place_id": 1264,
    "per_page" : 200,
    "fields": "taxon.id,taxon.name,taxon.rank,observed_on"
}


Next, gather these into a dataframe(?)

In [4]:
all_observations = []
page = 1

#for every page of observations, add results to all_observations list
while page <= 50:
    print(f"Downloading page {page}")

    params["page"] = page

    response = requests.get(url, params=params)
    data = response.json()

    observations = data["results"]

    if len(observations) == 0:
        break

    all_observations.extend(observations)

    page += 1


We'll test if this works by trying to print some of the results.

In [ ]:
for obs in all_observations:
    print(obs["taxon"]["name"])

Success! Let's see if I can use the code made previously to get these into a nice CSV (even though I know it's not yet the full dataset)

In [17]:
#collapsing the species into a checklist
species = set()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species.add(obs["taxon"]["name"])

print(species)

{'Megachile apicalis', 'Halictus ligatus', 'Andrena vicina', 'Bombus griseocollis', 'Coelioxys octodentatus', 'Calliopsis andreniformis', 'Megachile campanulae', 'Xylocopa virginica', 'Anthidium manicatum', 'Melissodes bimaculatus', 'Xenoglossa pruinosa', 'Bombus perplexus', 'Andrena erigeniae', 'Stelis louisae', 'Agapostemon virescens', 'Megachile sculpturalis', 'Augochlora pura', 'Triepeolus lunatus', 'Coelioxys alternatus', 'Andrena miserabilis', 'Habropoda laboriosa', 'Bombus pensylvanicus', 'Andrena wilkella', 'Bombus impatiens', 'Ptilothrix bombiformis', 'Triepeolus remigatus', 'Melissodes trinodis', 'Megachile rotundata', 'Megachile inimica', 'Lasioglossum coeruleum', 'Bombus citrinus', 'Megachile texana', 'Hylaeus leptocephalus', 'Chelostoma philadelphi', 'Pseudoanthidium nanum', 'Melissodes subillatus', 'Lasioglossum pectorale', 'Lasioglossum imitatum', 'Coelioxys sayi', 'Anthidium oblongatum', 'Osmia georgica', 'Osmia lignaria', 'Andrena milwaukeensis', 'Coelioxys coturnix', 

In [18]:
#count observations per species
species_counts = Counter()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species_counts[obs["taxon"]["name"]] += 1

print("\n".join(species_counts))




Melissodes denticulatus
Apis mellifera
Pseudoanthidium nanum
Xylocopa virginica
Bombus fervidus
Melissodes bimaculatus
Bombus impatiens
Megachile sculpturalis
Triepeolus lunatus
Bombus griseocollis
Halictus ligatus
Ptilothrix bombiformis
Bombus citrinus
Halictus confusus
Colletes latitarsis
Anthidium manicatum
Megachile texana
Megachile apicalis
Agapostemon virescens
Xenoglossa pruinosa
Bombus bimaculatus
Colletes thoracicus
Megachile rotundata
Megachile mendica
Hylaeus modestus
Andrena miserabilis
Habropoda laboriosa
Andrena vicina
Andrena erigeniae
Andrena hirticincta
Augochlora pura
Nomada placida
Megachile pusilla
Anthophora terminalis
Anthidium oblongatum
Dufourea novaeangliae
Megachile pugnata
Stelis louisae
Andrena wilkella
Osmia bucephala
Agapostemon sericeus
Coelioxys octodentatus
Megachile frigida
Triepeolus remigatus
Melissodes trinodis
Megachile inimica
Bombus perplexus
Andrena nuda
Andrena milwaukeensis
Colletes inaequalis
Andrena imitatrix
Coelioxys sayi
Melissodes agilis

In [19]:
#Converting this count into a table
species_df = pd.DataFrame(
    species_counts.items(),
    columns=["Species", "Observations"]
)

print(species_df)

#Sorting that dataframe
species_df = species_df.sort_values(
    by="Observations",
    ascending=False
)

print(species_df)

                    Species  Observations
0   Melissodes denticulatus            36
1            Apis mellifera          2568
2     Pseudoanthidium nanum             6
3        Xylocopa virginica           717
4           Bombus fervidus           176
..                      ...           ...
70         Megachile exilis             1
71     Coelioxys alternatus             1
72     Bombus pensylvanicus             1
73         Osmia cornifrons             1
74    Agapostemon splendens             1

[75 rows x 2 columns]
                  Species  Observations
6        Bombus impatiens          3081
1          Apis mellifera          2568
9     Bombus griseocollis          1427
3      Xylocopa virginica           717
15    Anthidium manicatum           208
..                    ...           ...
70       Megachile exilis             1
71   Coelioxys alternatus             1
72   Bombus pensylvanicus             1
73       Osmia cornifrons             1
74  Agapostemon splendens        

In [21]:
#Exporting the dataframe
species_df.to_csv("C:/Users/brigu/Documents/_Fordham/_PhD_RESEARCH/bee-project-coding/data/processed/bee_species_test3.csv", index=False)

Okay! That got me a nice CSV summarizing those 10,000 observations. Next steps:

    - get around the 10,000 item limit (or is it a 50-page limit?)
    - get Bronx counts as well
    - merge these into one list, with Manhattan, Bronx, and combined columns (sorted by combined column)
    - maybe even get it to automatically populate it with genus and family columns??